Visualize the statistics.

In [ ]:
%cd -q ..
from impl.utils.visualization import Visualizer

Visualizer.visualize_in_one("out/solutions/statistics-1715715500855.pickle", show=True)
%cd -q test

Run the best complete solutions.

In [ ]:
%cd -q ..
from impl.scenario.simulation_runner import run_solutions

run_solutions("out/solutions/solutions-1715028728116.pickle", top=1)
%cd -q test

Visualize the violation.

In [ ]:
%cd -q ..
import math
import matplotlib.pyplot as plt
import pandas as pd
from tslearn.metrics import dtw_path
from impl.problem import mr_set

source = pd.read_csv("out/results/top1_source.csv")
source.set_index(source.columns[0], inplace=True)
follow_up = pd.read_csv("out/results/top1_follow-up.csv")
follow_up.set_index(follow_up.columns[0], inplace=True)

field = mr_set.field()
if field == "velocity":
    func = lambda row: math.sqrt(row.velocity_x ** 2 + row.velocity_y ** 2)
    source[field] = source.apply(func, axis=1, result_type="reduce")
    follow_up[field] = follow_up.apply(func, axis=1, result_type="reduce")
print("is_violated: {}, extent: {}".format(*mr_set.is_violated(source, follow_up)))
%cd -q test

offset = 0
matches, _ = dtw_path(source[field], follow_up[field])
source["diff"] = source[field] - follow_up[field]
source[field] -= offset
fig, axs = plt.subplots()
axs.plot(list(source.index.values), list(source["diff"]), "o--C1", label="diff")
axs.plot(list(source.index.values), list(source[field]), "-C2", label="source")
axs.plot(list(follow_up.index.values), list(follow_up[field]), "-C3", label="follow up")
for i, j in matches:
    axs.plot((source.index.values[i], follow_up.index.values[j]),
             (source.loc[source.index.values[i], field], follow_up.loc[follow_up.index.values[j], field]),
             "--", color="gray")
axs.legend()
fig.tight_layout()
plt.show()